# 🔬 Урок 52: Seaborn + Plotly — Сучасна аналітична візуалізація

> **Мета уроку:** Перейти від статичних графіків до **статистично осмисленої** та **інтерактивної** візуалізації. Навчитися обирати правильний інструмент для правильного завдання.

---

## 🗺️ Зміст уроку

1. Три рівні візуалізації: Matplotlib → Seaborn → Plotly
2. Seaborn: Boxplot — аномалії та розподіл
3. Seaborn: Heatmap — кореляційний аналіз
4. Seaborn: Pairplot — розвідувальний аналіз (EDA)
5. Plotly: інтерактивні лінійні графіки
6. Plotly: hover, zoom, часові ряди з RangeSlider

---

**Датасети:**
- 🌾 `wfp_food_prices_ukr.csv` — ціни на продукти харчування (WFP)
- 💱 `exchange-rates_ukr.csv` — курс UAH/USD
- 📉 `poverty_ukr.csv` — соціально-економічні показники
- 🌍 `global-market-monitor.csv` — глобальний моніторинг ринку

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = 'data'

# --- Завантаження датасетів ---
df_food = pd.read_csv(
    os.path.join(DATA_DIR, 'wfp_food_prices_ukr.csv'),
    usecols=['date', 'admin1', 'market', 'category', 'commodity', 'unit', 'currency', 'price'],
    parse_dates=['date']
)
df_rates = pd.read_csv(
    os.path.join(DATA_DIR, 'exchange-rates_ukr.csv'),
    usecols=['StartDate', 'Year', 'Months', 'Value'],
    parse_dates=['StartDate']
)

# --- 1. Завантажуємо ціни на продукти харчування ---
# Спочатку подивимося, скільки рядків у файлі
df_rates.columns = ['date', 'year', 'month', 'uah_per_usd']

# Гривневі ціни + допоміжні колонки
df_uah = df_food.query("currency == 'UAH'").copy()
df_uah['рік']   = df_uah['date'].dt.year
df_uah['місяць'] = df_uah['date'].dt.month
df_rates['рік']  = df_rates['date'].dt.year

df_global = pd.read_csv(os.path.join(DATA_DIR, 'global-market-monitor.csv'))
df_global['date'] = pd.to_datetime(df_global['Date'])

print(f"✅ Продовольчі ціни:   {df_food.shape[0]:,} рядків")
print(f"✅ Курс UAH/USD:        {df_rates.shape[0]} рядків")
print(f"✅ Seaborn версія:      {sns.__version__}")

In [ ]:
print(df_global.columns.tolist())

---

## 1. Три рівні візуалізації 🏛️ <a id='1-comparison'></a>

### Вибір бібліотеки — це архітектурне рішення

| | Matplotlib | Seaborn | Plotly |
|---|---|---|---|
| **Рівень** | Low-level рушій | High-level надбудова над Matplotlib | Веб-рушій (JSON → D3.js) |
| **Вихід** | Статична картинка (PNG, PDF) | Статична картинка | Інтерактивний HTML/JS |
| **Код** | Багато, повний контроль | Мало (1–2 рядки) | Мало (Plotly Express) |
| **Pandas-інтеграція** | Базова | ✅ Нативна | ✅ Нативна |
| **Статистика** | Вручну | ✅ Вбудована | Базова |
| **Hover / Zoom** | ❌ | ❌ | ✅ |

### Коли що використовувати?

```
Matplotlib  → наукові публікації, нестандартні графіки, власні інструменти
Seaborn     → розвідувальний аналіз (EDA) у Jupyter, статистичні звіти
Plotly      → бізнес-дашборди, веб-застосунки (Dash), презентації
```

> 💡 **Головна ідея:** Не існує «найкращої» бібліотеки. Є правильна бібліотека для конкретного завдання та аудиторії.

In [ ]:
# --- Той самий графік: Matplotlib vs Seaborn vs Plotly ---
# Один рядок даних — три різні рівні коду

bread_yearly = (
    df_uah.query("commodity.str.contains('Bread') and admin1.isna()")
    .groupby('рік')['price'].mean().round(2).reset_index()
)

# --- Рівень 1: Matplotlib (явний контроль) ---
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(bread_yearly['рік'], bread_yearly['price'], color='steelblue', linewidth=2, marker='o', markersize=4)
ax.set_title('Matplotlib: повний контроль')
ax.set_xlabel('Рік'); ax.set_ylabel('UAH')
ax.grid(True, linestyle=':', alpha=0.4)
fig.tight_layout(); plt.show()

# --- Рівень 2: Seaborn (стиль + статистика) ---
fig, ax = plt.subplots(figsize=(8, 3))
sns.lineplot(data=bread_yearly, x='рік', y='price', ax=ax, color='steelblue', linewidth=2, marker='o')
ax.set_title('Seaborn: стиль + автоматична статистика')
fig.tight_layout(); plt.show()

# --- Рівень 3: Plotly (інтерактивний) ---
fig_px = px.line(
    bread_yearly, x='рік', y='price',
    title='Plotly: інтерактивний (hover для деталей)',
    labels={'рік': 'Рік', 'price': 'Середня ціна хліба (UAH)'},
    markers=True
)
fig_px.update_layout(height=350)
fig_px.show()

---

## 2. Seaborn: Boxplot — аномалії та розподіл 📦 <a id='2-boxplot'></a>

### Що таке «ящик з вусами»?

Boxplot — стандарт для **пошуку аномалій** та **порівняння розподілів** між категоріями.

```
         ┌─────────────────┐
  ───────┤  Q1    MED   Q3 ├────────  ← IQR (де 50% даних)
         └─────────────────┘
  ↑                               ↑
  мін (≤ Q1 − 1.5×IQR)     макс (≤ Q3 + 1.5×IQR)
  
  ● ●   ← точки поза вусами = викиди (outliers)
```

| Елемент | Що показує |
|---|---|
| **Ящик (Box)** | Міжквартильний розмах IQR = Q3 − Q1 (середні 50% даних) |
| **Лінія медіани** | 50-й процентиль — стійкіша до викидів, ніж середнє |
| **Вуса (Whiskers)** | Дані до 1.5 × IQR від ящика |
| **Точки за вусами** | Статистичні аномалії (outliers) |

> 💡 **Seaborn vs Matplotlib:** `sns.boxplot()` — один рядок. У чистому Matplotlib це десятки рядків коду для розрахунку квартилів і малювання прямокутників.

In [ ]:
# --- Підготовка: ціни 4 популярних товарів ---
selected = ['Bread (wheat)', 'Potatoes', 'Oil (sunflower)', 'Milk']
df_box = df_uah.query("commodity in @selected and price < 200").copy()

# --- Базовий Boxplot ---
fig, ax = plt.subplots(figsize=(11, 5))

sns.boxplot(
    data=df_box,
    x='commodity',
    y='price',
    palette='Set2',        # вбудована кольорова палітра Seaborn
    width=0.5,
    fliersize=3,           # розмір точок-викидів
    linewidth=1.5,
    ax=ax
)

ax.set_title('Розподіл цін на основні продукти (UAH) — всі роки', fontsize=13, fontweight='bold')
ax.set_xlabel('Товар')
ax.set_ylabel('Ціна (UAH)')
ax.grid(True, axis='y', linestyle=':', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
plt.show()
print("💡 Точки над 'вусами' — статистичні аномалії (наприклад, ринкові стрибки цін)")

In [ ]:
# --- Boxplot + stripplot: показуємо і розподіл, і окремі точки ---
# hue='епоха' — розкладаємо на два часових проміжки

df_box['епоха'] = df_box['рік'].apply(lambda y: 'до 2022' if y < 2022 else 'з 2022')

fig, ax = plt.subplots(figsize=(13, 6))

sns.boxplot(
    data=df_box,
    x='commodity', y='price',
    hue='епоха',            # розділяє ящики по підгрупах
    palette={'до 2022': '#5ba85f', 'з 2022': '#e07b39'},
    width=0.6,
    fliersize=2,
    linewidth=1.2,
    ax=ax
)

ax.set_title('Розподіл цін: до та після 2022 року', fontsize=13, fontweight='bold')
ax.set_xlabel('Товар')
ax.set_ylabel('Ціна (UAH)')
ax.legend(title='Епоха', loc='upper left')
ax.grid(True, axis='y', linestyle=':', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
plt.show()
print("💡 hue='епоха' — Seaborn автоматично розбиває ящики та додає легенду. У Matplotlib — десятки рядків коду.")

In [ ]:
# --- Violin plot: розширена версія boxplot ---
# Violin = boxplot + KDE (форма розподілу по ширині)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ліворуч: boxplot
sns.boxplot(
    data=df_box, x='commodity', y='price',
    palette='Set2', ax=axes[0], fliersize=2
)
axes[0].set_title('Boxplot: квартилі та викиди', fontweight='bold')
axes[0].set_ylabel('Ціна (UAH)')
axes[0].tick_params(axis='x', rotation=15)

# Праворуч: violin
sns.violinplot(
    data=df_box, x='commodity', y='price',
    palette='Set2', ax=axes[1],
    inner='quartile',  # показує квартилі всередині скрипки
    cut=0
)
axes[1].set_title('Violinplot: boxplot + форма розподілу (KDE)', fontweight='bold')
axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=15)

fig.suptitle('Boxplot vs Violin: два рівні деталізації розподілу', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()
print("💡 Violin розповідає більше: ширина в кожній точці = густота даних = KDE")

### 🎯 Міні-завдання 1

1. Побудуй boxplot розподілу цін для 5 різних **регіонів** (`admin1`) у 2023 році для товару `Bread (wheat)`
2. Який регіон має найбільше «вусів» (найбільший розкид цін)?
3. Порівняй `boxplot` та `violinplot` — в якому випадку кожен з них інформативніший?

> **Очікуваний інсайт:** Розподіл цін між регіонами нерівномірний — прифронтові регіони або тимчасово окуповані мають аномально високі або низькі ціни. Boxplot робить ці аномалії видимими одразу.

In [ ]:
# 🎯 Ваш код тут:


---

## 3. Seaborn: Heatmap — кореляційний аналіз 🌡️ <a id='3-heatmap'></a>

### Що таке кореляційна матриця?

Кореляція Пірсона вимірює **лінійний зв'язок** між двома змінними:

```
+1.0  →  ідеальна пряма залежність (більше X → більше Y)
 0.0  →  зв'язку немає
-1.0  →  ідеальна зворотна залежність (більше X → менше Y)
```

`df.corr()` повертає матрицю N×N — нудну таблицю чисел. `sns.heatmap()` перетворює її на **кольорове полотно**, де:
- 🔴 Темно-червоний = сильна позитивна кореляція
- ⚪ Білий/блідий = відсутність зв'язку
- 🔵 Темно-синій = сильна негативна кореляція

### Використання в аналітиці

Heatmap — перший крок перед **побудовою регресійних моделей**: визначаємо, які змінні пов'язані, щоб уникнути **мультиколінеарності** (коли два предиктори говорять одне й те саме).

In [ ]:
# --- Підготовка: зведена таблиця цін по роках і категоріях ---
# Pivot: рядки = роки, стовпці = категорії, значення = середня ціна

pivot_price = (
    df_uah
    .groupby(['рік', 'category'])['price']
    .mean()
    .unstack()
    .round(2)
)

# Скорочуємо назви категорій для читабельності
pivot_price.columns = [
    col.replace(' and ', ' & ').replace('cereals & tubers', 'Зернові')
       .replace('vegetables & fruits', 'Овочі/фрукти')
       .replace('oil & fats', 'Олія/жири')
       .replace('meat, fish & eggs', "М'ясо/риба")
       .replace('milk & dairy', 'Молоко/молочні')
       .replace('miscellaneous food', 'Інше')
    for col in pivot_price.columns
]

# Матриця кореляцій між категоріями
corr_matrix = pivot_price.corr()

print("Форма зведеної таблиці:", pivot_price.shape)
print("\nМатриця кореляцій (перші 3×3):")
corr_matrix.iloc[:3, :3].round(3)

In [ ]:
# --- Heatmap кореляцій між категоріями цін ---
fig, ax = plt.subplots(figsize=(9, 7))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # маскуємо верхній трикутник (дублікат)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,            # показуємо числові значення
    fmt='.2f',             # формат: 2 знаки після крапки
    cmap='coolwarm',       # дивергентна палітра: синій-білий-червоний
    center=0,              # нуль = білий
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor='white',
    square=True,
    ax=ax
)

ax.set_title('Кореляція між категоріями продовольчих цін (1996–2024)',
             fontsize=13, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

fig.tight_layout()
plt.show()
print("\n💡 Темно-червоний = категорії ростуть в ціні разом (системна інфляція)")
print("   Маска верхнього трикутника прибирає дублікати — читати легше")

In [ ]:
# --- Heatmap: ціна хліба по місяцях і роках ---
# Як змінювалась сезонність після 2014 і 2022?

bread_pivot = (
    df_uah
    .query("commodity.str.contains('Bread') and admin1.isna() and рік >= 2014")
    .groupby(['рік', 'місяць'])['price']
    .mean()
    .unstack()
    .round(1)
)
bread_pivot.columns = ['Січ','Лют','Бер','Кві','Тра','Чер','Лип','Сер','Вер','Жов','Лис','Гру']

fig, ax = plt.subplots(figsize=(14, 7))

sns.heatmap(
    bread_pivot,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',          # жовто-оранжево-червона палітра
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Ціна (UAH)', 'shrink': 0.7},
    ax=ax
)

ax.set_title('Ціна хліба (UAH): рік × місяць — теплова карта сезонності',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Місяць')
ax.set_ylabel('Рік')

fig.tight_layout()
plt.show()
print("💡 Темніші рядки = роки з вищими цінами. Колонки показують сезонність.")

### 🎯 Міні-завдання 2

1. Побудуй heatmap кореляції між **курсом UAH/USD** та цінами по категоріях (підказка: об'єднай датасети через `pd.merge`, потім `.corr()`)
2. Яка категорія найбільше корелює з курсом долара?
3. Зміни `cmap` на `'RdYlGn_r'` — як зміниться читабельність?

> **Очікуваний інсайт:** Категорії олії/жирів та імпортованих товарів покажуть найвищу кореляцію з курсом долара — це прямий вплив девальвації на споживчі ціни.

In [ ]:
# 🎯 Ваш код тут:


---

## 4. Seaborn: Pairplot — розвідувальний аналіз (EDA) 🔭 <a id='4-pairplot'></a>

### Pairplot — найпотужніший EDA-інструмент

Один виклик `sns.pairplot(df)` будує матрицю N×N графіків:

```
         X1         X2         X3
X1  [KDE/hist] [scatter] [scatter]
X2  [scatter] [KDE/hist] [scatter]
X3  [scatter] [scatter] [KDE/hist]
     ↑ діагональ       ↑ поза діагоналлю
     розподіл кожної    зв'язок між парами
     змінної окремо
```

### Суперсила: параметр `hue`

Передаємо категоріальну колонку → Seaborn автоматично фарбує точки за групами. Це дозволяє **побачити кластери** та зрозуміти, як категорія впливає на зв'язки між числовими змінними.

In [ ]:
# --- Підготовка: зведена таблиця для pairplot ---
# Беремо 4 ключові товари по роках (National Average)

key_foods = ['Bread (wheat)', 'Potatoes', 'Oil (sunflower)', 'Milk']

df_pairplot = (
    df_uah
    .query("commodity in @key_foods and admin1.isna()")
    .pivot_table(
        index='date',
        columns='commodity',
        values='price'
    )
    .reset_index()
)
df_pairplot.columns.name = None

df_rates['date'] = pd.to_datetime(df_rates['date'])


# Об'єднуємо з курсом UAH/USD
df_pairplot['month'] = df_pairplot['date'].dt.to_period('M')
df_rates['month'] = df_rates['date'].dt.to_period('M')

df_pairplot = df_pairplot.merge(
    df_rates[['month', 'uah_per_usd']],
    on='month',
    how='inner'
)
df_pairplot['date'] = df_pairplot['month'].dt.to_timestamp()
df_pairplot['епоха'] = df_pairplot['date'].dt.year.apply(
    lambda y: 'до 2022' if y < 2022 else 'з 2022'
)

print(f"Таблиця для pairplot: {df_pairplot.shape}")
df_pairplot.head()

In [ ]:
# --- Pairplot: всі попарні зв'язки між цінами та курсом ---
numeric_cols = ['Bread (wheat)', 'Potatoes', 'Oil (sunflower)', 'Milk', 'uah_per_usd']

g = sns.pairplot(
    df_pairplot[numeric_cols + ['епоха']],
    hue='епоха',                     # розфарбовуємо по епохах
    palette={'до 2022': '#3b82c4', 'з 2022': '#e07b39'},
    diag_kind='kde',                 # на діагоналі — KDE замість гістограми
    plot_kws={'alpha': 0.7, 's': 50},
    diag_kws={'fill': True, 'alpha': 0.4}
)

g.fig.suptitle(
    'Pairplot: попарні зв\'язки між цінами та курсом UAH/USD',
    fontsize=13, fontweight='bold', y=1.02
)

# Скорочені назви осей
labels = ['Хліб', 'Картопля', 'Олія', 'Молоко', 'Курс UAH']
for ax, label in zip(g.axes[-1], labels):
    ax.set_xlabel(label, fontsize=9)
for ax, label in zip(g.axes[:, 0], labels):
    ax.set_ylabel(label, fontsize=9)

plt.tight_layout()
plt.show()

# 📊 Аналіз взаємозв’язків між цінами на продукти та курсом UAH/USD

## 1. Загальна постановка задачі

Метою аналізу є виявлення:

* взаємозв’язків між цінами на ключові продукти
* впливу валютного курсу (UAH/USD)
* змін структури економічної системи до та після 2022 року

Аналіз проведено на основі **місячних агрегованих даних (National Average)** для товарів:

* Хліб (Bread)
* Картопля (Potatoes)
* Олія (Sunflower oil)
* Молоко (Milk)

---

## 2. Аналіз залежності: курс ↔ ціни

### 🔹 Загальний висновок

Курс UAH/USD демонструє **позитивний зв’язок із цінами на продукти**:

> 📈 Зростання курсу → зростання цін

---

### 🟢 Хліб ↔ Курс

* Спостерігається **чітка позитивна залежність**
* Ціни на хліб зростають разом із девальвацією гривні

📌 Інтерпретація:

* хліб частково залежить від імпортних компонентів (паливо, логістика, добрива)
* ефект передачі інфляції (inflation pass-through)

---

### 🔴 Олія ↔ Курс (найсильніший зв’язок)

* Найбільш виражена лінійна залежність

📌 Інтерпретація:

* олія є **експортно-орієнтованим товаром**
* її ціна формується на глобальному ринку
* курс напряму впливає на внутрішню ціну

---

### 🟡 Молоко ↔ Курс

* Зв’язок слабший

📌 Інтерпретація:

* переважно локальне виробництво
* менша залежність від імпорту

---

### 🟡 Картопля ↔ Курс

* Найслабший зв’язок

📌 Інтерпретація:

- * локальний продукт

- * ключовий фактор — **сезонність**, а не курс

---

## 3. Взаємозв’язки між продуктами

### 🔥 Ключове спостереження

Ціни на продукти демонструють **взаємну кореляцію**:

* Олія ↔ Хліб
* Олія ↔ Молоко
* Хліб ↔ Картопля

📌 Інтерпретація:

> ❗ Існує **загальний інфляційний фактор**, що впливає на всі продукти одночасно

---

## 4. Аналіз розподілів (діагональ pairplot)

### 🔹 Хліб

* Спостерігається **двомодальний розподіл (2 піки)**

### 🔹 Курс UAH/USD

* Також має два чіткі режими

---

## 💥 Ключовий інсайт

> ❗ Дані описують **не одну систему, а дві різні економічні фази**

---

## 5. Структурний злам у 2022 році

### 🔍 Спостереження

У scatter-графіках видно:

* точки формують **дві окремі групи**
* залежності виглядають “зламаними”

---

### 🧠 Інтерпретація

- > ❗ Залежності **не є лінійними у часі**
- > ❗ Відбулась **зміна економічного режиму**

---

### 📊 Переклад на аналітичну мову

| Фактор   | Висновок                  |
| -------- | ------------------------- |
| Курс     | впливає на всі ціни       |
| Олія     | найбільш чутлива до курсу |
| Картопля | найменш чутлива           |
| 2022     | структурний злам системи  |

---

## 6. Системний висновок

Після 2022 року:

* кореляції між змінними посилились
* ціни рухаються більш синхронно
* система стала більш “зчепленою”

---

### 🧠 Це означає:

> 🔥 **перехід від відносно незалежного ринку → до системно пов’язаної економіки**

---

## 7. Узагальнення

Аналіз показує:

1. Валютний курс є ключовим драйвером цін
2. Різні товари мають різну чутливість до курсу
3. Існує загальний інфляційний фактор
4. 2022 рік є точкою структурного зламу
5. Економічна система перейшла у новий режим

---

## 🚀 Подальші кроки аналізу

* аналіз лагів (курс → ціни із затримкою)
* регресійні моделі (оцінка впливу)
* аналіз причинності (Granger causality)
* кластеризація товарів за поведінкою

---

# 🧠 Фінальний висновок

> 📊 Дані показують не просто зміну цін
> а **перебудову економічної системи під впливом зовнішнього шоку**


In [ ]:
df_pairplot.groupby('епоха')[numeric_cols].corr()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for epoch in df_pairplot['епоха'].unique():
    corr = (
        df_pairplot[df_pairplot['епоха'] == epoch][numeric_cols]
        .corr()
    )

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        corr,
        annot=True,
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        center=0
    )

    plt.title(f'Кореляція змінних ({epoch})')
    plt.show()

In [ ]:
# --- Аналіз кореляцій з коментарями ---
corr_full = df_pairplot[numeric_cols].corr().round(3)

print("=== Матриця кореляцій ===")
print(corr_full)

# Топ-5 найсильніших кореляцій (окрім діагоналі)
corr_pairs = (
    corr_full
    .where(np.tril(np.ones(corr_full.shape), k=-1).astype(bool))  # нижній трикутник
    .stack()
    .reset_index()
)
corr_pairs.columns = ['Змінна 1', 'Змінна 2', 'Кореляція']
corr_pairs['|r|'] = corr_pairs['Кореляція'].abs()

print("\n=== Топ-5 найсильніших зв'язків ===")
print(corr_pairs.nlargest(5, '|r|')[['Змінна 1', 'Змінна 2', 'Кореляція']].to_string(index=False))
print("\n💡 Помаранчеві точки (з 2022) відірвались від синього хмари — структурний зсув у цінах")

### 🎯 Міні-завдання 3

1. Додай до `df_pairplot` колонку з відносним зростанням ціни хліба (% до базового 2014 року) і включи її в pairplot
2. Знайди пару змінних з **найслабшою** кореляцією — як це виглядає на scatter?
3. Поміняй `diag_kind='kde'` на `diag_kind='hist'` — що змінилось у читабельності?

> **Очікуваний інсайт:** Pairplot показує, що після 2022 (помаранчеві точки) зв'язки між цінами змінились — вони утворюють окремий кластер, що свідчить про структурний зсув у продовольчій системі.

In [ ]:
# 🎯 Ваш код тут:


---

## 5. Plotly: інтерактивні лінійні графіки 🖱️ <a id='5-plotly-line'></a>

### Архітектура Plotly під капотом

```
Python (Plotly) → JSON структура → D3.js / WebGL → HTML / браузер
```

Ваш Python-код генерує **JSON-опис** графіка. Браузер рендерить його через JavaScript. Саме тому Plotly-графіки інтерактивні — це справжні веб-компоненти.

### Plotly Express vs Graph Objects

| | `plotly.express` (px) | `plotly.graph_objects` (go) |
|---|---|---|
| Стиль | High-level, 1 рядок | Low-level, повний контроль |
| Аналог | Seaborn | Matplotlib |
| Коли | Швидкий EDA | Складні кастомні дашборди |

### Що робить Plotly особливим для бізнесу?

> Ти перетворюєш споживача інформації з **пасивного глядача** статичного PDF на **активного дослідника**, який сам фільтрує, масштабує і досліджує дані.

In [ ]:

# --- Plotly Express: інтерактивний лінійний графік ---
# Спробуй: hover на точки, виділи область мишею (zoom), подвійний клік — скидання

bread_monthly = (
    df_uah
    .query("commodity.str.contains('Bread') and admin1.isna()")
    .groupby('date')['price'].mean().round(2)
    .reset_index()
)

bread_monthly['date'] = pd.to_datetime(bread_monthly['date'])
fig = px.line(
    bread_monthly,
    x='date',
    y='price',
    title='Ціна хліба в Україні (2014–2024) — інтерактивно 🖱️',
    labels={'date': 'Дата', 'price': 'Ціна (UAH)'},
    color_discrete_sequence=['#1a6faf']
)


fig.show()
print("💡 Спробуй: виділи мишею 2022–2023 рік → автоматичний zoom")

In [ ]:
# --- Кілька серій + hover_data ---
# hover_data = додаткові поля в підказці

multi_foods = ['Bread (wheat)', 'Potatoes', 'Oil (sunflower)']

df_multi = (
    df_uah
    .query("commodity in @multi_foods and admin1.isna()")
    .groupby(['date', 'commodity'])['price']
    .mean().round(2)
    .reset_index()
)

fig = px.line(
    df_multi,
    x='date', y='price',
    color='commodity',        # кожна серія — свій колір
    title='Динаміка цін на 3 товари — hover для деталей',
    labels={'date': 'Дата', 'price': 'Ціна (UAH)', 'commodity': 'Товар'},
    color_discrete_map={
        'Bread (wheat)': '#e07b39',
        'Potatoes': '#5ba85f',
        'Oil (sunflower)': '#3b82c4'
    }
)



fig.update_layout(
    height=450,
    hovermode='x unified',
    legend=dict(title='Товар', orientation='h', y=-0.15),
    plot_bgcolor='black',
    xaxis=dict(showgrid=True, gridcolor='#eee'),
    yaxis=dict(showgrid=True, gridcolor='#eee')
)

fig.show()

### 🎯 Міні-завдання 4

1. Побудуй інтерактивний Plotly scatter plot між курсом UAH/USD та ціною хліба (кожна точка = 1 рік, `hover_data=['рік']`)
2. Додай лінію тренду через `trendline='ols'` у `px.scatter()` — що вона показує?
3. Зроби так, щоб крапки були різного кольору до/після 2022

> **Очікуваний інсайт:** На Plotly scatter ти можеш навести курсор на кожну точку та побачити точний рік і значення — це набагато зручніше для пояснення закономірностей замовнику, ніж статична картинка.

In [ ]:
# 🎯 Ваш код тут:


---

## 6. Plotly: hover, zoom, часові ряди з RangeSlider 📊 <a id='6-plotly-timeseries'></a>

### Проблема статичних часових рядів

Коли дані охоплюють 10+ років, статичний графік приховує деталі. У Plotly аналітик може:

| Дія | Що відкриває |
|---|---|
| **Hover** | Точне значення, дата, назва категорії |
| **Zoom** | Деталізація конкретного місяця або кварталу |
| **RangeSlider** | Динамічне переміщення вікна перегляду |
| **RangeSelector** | Кнопки «6M», «1Y», «2Y», «All» |
| **Legend click** | Сховати/показати окрему серію |

### `graph_objects` — низький рівень Plotly

Для складних графіків (subplots, кастомні tooltip, змішані типи) використовуємо `go.Figure()` — аналог OOP-стилю в Matplotlib.

In [ ]:
# --- Plotly: глобальне порівняння — heatmap з hover ---
# Середнє річне зростання цін по країнах

top_countries = (
    df_global
    .query("YoYChangeMonth.notna()")
    ['CountryName'].value_counts()
    .head(11).index.tolist()
)

heat_data = (
    df_global
    .query("CountryName in @top_countries and YoYChangeMonth.notna()")
    .assign(рік=lambda x: pd.to_datetime(x['Date']).dt.year)
    .groupby(['CountryName', 'рік'])['YoYChangeMonth']
    .mean().round(1)
    .unstack()
)

fig = px.imshow(
    heat_data,
    color_continuous_scale='RdYlGn_r',  # зелений = низьке, червоний = високе
    color_continuous_midpoint=0,
    aspect='auto',
    title='Річне зростання цін на продовольство по країнах (%, YoY)',
    labels=dict(x='Рік', y='Країна', color='Зміна (%)')
)

fig.update_layout(height=500, coloraxis_colorbar=dict(title='YoY %'))
fig.show()
print("💡 Hover на клітинку → точна країна, рік та відсоток зміни цін")

In [ ]:
df_proc = (
    df_global
    .query("YoYChangeMonth.notna()")
    .assign(рік=lambda x: pd.to_datetime(x['Date']).dt.year)
)

heat_data = (
    df_proc
    .groupby(['CountryName', 'рік'])['YoYChangeMonth']
    .median()
    .unstack()
)

# залишаємо країни з ≥4 роками
heat_data = heat_data.dropna(axis=0, thresh=4)

In [ ]:
ranking = heat_data.median(axis=1).sort_values()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=("📊 Динаміка (Heatmap)", "🏆 Рейтинг країн")
)
fig.add_trace(
    go.Heatmap(
        z=heat_data.values,
        x=heat_data.columns,
        y=heat_data.index,
        colorscale='RdYlGn_r',
        zmid=0,
        colorbar=dict(title="YoY %")
    ),
    row=1, col=1
)
fig.add_trace(
    go.Bar(
        x=ranking.values,
        y=ranking.index,
        orientation='h',
        marker=dict(color=ranking.values, colorscale='RdYlGn_r'),
    ),
    row=1, col=2
)
fig.update_layout(
    height=600,
    title="🌍 Аналіз інфляції продовольства: динаміка + рейтинг",
    showlegend=False
)

fig.show()

In [ ]:
stats = pd.DataFrame({
    'median_yoy': heat_data.median(axis=1),
    'volatility': heat_data.std(axis=1)
})

In [ ]:
fig = px.scatter(
    stats,
    x='median_yoy',
    y='volatility',
    text=stats.index,
    title="📊 Стабільність vs Інфляція",
    labels={
        'median_yoy': 'Медіанна інфляція (%)',
        'volatility': 'Волатильність (std)'
    }
)

fig.update_traces(textposition='top center')
fig.show()

### 🎯 Фінальне завдання

Створи **власний інтерактивний аналітичний звіт** у Plotly:

1. **Вибери свою гіпотезу** — наприклад: *«Ціни на олію в Україні зростають швидше, ніж у сусідніх країнах»*
2. Підготуй дані (pandas: фільтрація, groupby, merge)
3. Побудуй **3 Plotly-графіки** для доведення або спростування гіпотези:
   - Line plot з RangeSlider (динаміка у часі)
   - Bar chart або heatmap (порівняння)
   - Scatter з hover (кореляція)
4. Додай заголовки, анотації ключових подій і кастомний hover

> **Очікуваний інсайт:** Інтерактивна візуалізація — це не просто «красивіше». Це інший спосіб мислення про дані: замість того, щоб готувати окремий графік на кожне питання, ти даєш інструмент, яким аудиторія може самостійно відповідати на питання.

In [ ]:
# 🎯 Ваш аналітичний звіт тут:


---

## 🏁 Підсумок уроку

### Фінальна таблиця вибору інструменту

| Завдання | Інструмент | Чому |
|---|---|---|
| Публікація / PDF / друк | Matplotlib | Статичний, повний контроль |
| EDA, розподіли, кореляції | Seaborn | 1–2 рядки коду, вбудована статистика |
| Аномалії в категоріях | `sns.boxplot()` | Квартилі + викиди автоматично |
| Зв'язки між змінними | `sns.heatmap()` + `pairplot()` | Миттєве читання кореляцій |
| Бізнес-звіт / дашборд | Plotly | Hover, zoom, RangeSlider |
| Веб-застосунок | Plotly + Dash | JSON → D3.js, React UI |

### Архітектурний ланцюжок

```
Matplotlib  ←──── base engine ────→  Seaborn (статистика)
                                           │
                                    Pandas .plot()

Plotly  ────  Python → JSON → D3.js/WebGL → HTML  ────→  Dash
```

### Наступний крок

У **Уроці 53** ми беремо Plotly і будуємо повноцінний **Dash-дашборд** — вебзастосунок з фільтрами, dropdown та real-time оновленням!

---
*📚 Урок 52 — Модуль 3: Dash Data Analysis & Visualization*